# BusTimer EDA

Connects to the Postgres database and pulls sample data from every table.

Loads credentials from `../../.env` (project root). The `.env` file must define:

- `POSTGRES_DB`
- `POSTGRES_USER`
- `POSTGRES_PW`

Host and port are hardcoded to `100.111.121.51:5432`.


In [1]:
import os
import pathlib

import polars as pl
import psycopg2
import requests
from psycopg2.extras import RealDictCursor

# Load .env from project root
env_path = pathlib.Path("../../.env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, _, value = line.partition("=")
            os.environ.setdefault(key.strip(), value.strip())

CONNECTION_PARAMS = {
    "host": "100.111.121.51",
    "port": 5433,
    "database": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PW"],
}

API_BASE = "http://100.111.121.51:8081"


def query(sql: str, params=None) -> pl.DataFrame:
    with (
        psycopg2.connect(**CONNECTION_PARAMS) as conn,
        conn.cursor(cursor_factory=RealDictCursor) as cur,
    ):
        cur.execute(sql, params)
        rows = cur.fetchall()
    return pl.DataFrame([dict(r) for r in rows])


print("Connected to", CONNECTION_PARAMS["host"])

Connected to 100.111.121.51


## Table: `calendar`

Service schedules — which days of the week each service runs.


In [2]:
calendar = query("SELECT * FROM calendar")
print(calendar.shape)
calendar.head(10)

(133, 8)


service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday
str,i64,i64,i64,i64,i64,i64,i64
"""Daily-3""",1,1,1,1,1,1,1
"""Daily-4""",1,1,1,1,1,1,1
"""Daily-5""",1,1,1,1,1,1,1
"""Daily-6""",1,1,1,1,1,1,1
"""ExSat-2""",1,1,1,1,1,0,1
"""ExSat-3""",1,1,1,1,1,0,1
"""ExSat-4""",1,1,1,1,1,0,1
"""ExSun-2""",1,1,1,1,1,1,0
"""ExSun-3""",1,1,1,1,1,1,0


## Table: `stops`

Bus stop locations and metadata.


In [3]:
stops = query("SELECT * FROM stops")
print(stops.shape)
stops.head(10)

(211, 6)


stop_id,location_type,stop_code,stop_lat,stop_lon,stop_name
str,i64,str,f64,f64,str
"""7890-7ac9c7e4""",0,"""7890""",-36.8567,174.81022,"""Paritai Drive"""
"""7309-2c09a359""",0,"""7309""",-36.84897,174.79291,"""Parnell Baths"""
"""8074-957bc7e4""",0,"""8074""",-36.88334,174.80648,"""Grand View Road"""
"""7108-87efa451""",0,"""7108""",-36.84815,174.7524,"""Franklin Road"""
"""8017-1a66e258""",0,"""8017""",-36.86701,174.70765,"""Alberta Street"""
"""7449-9327eaba""",0,"""7449""",-36.87558,174.84602,"""Cotton Street"""
"""7838-191c25de""",0,"""7838""",-36.86541,174.84388,"""299 Kohimarama Road"""
"""8125-0b23e96e""",0,"""8125""",-36.8679,174.7197,"""Auckland Zoo"""
"""7431-a753663c""",0,"""7431""",-36.87692,174.82537,"""Remuera Road/Meadowbank Road"""


## Table: `segments`

Point-to-point segments between stops.


In [4]:
segments = query("""
    SELECT s.*, 
           st.stop_name AS start_stop_name,
           en.stop_name AS end_stop_name
    FROM segments s
    JOIN stops st ON st.stop_id = s.start_stop_id
    JOIN stops en ON en.stop_id = s.end_stop_id
""")
print(segments.shape)
segments.head(10)

(212, 5)


segment_id,start_stop_id,end_stop_id,start_stop_name,end_stop_name
str,str,str,str,str
"""7221-7219""","""7221-b28d532b""","""7219-06392a6a""","""Ponsonby Methodist Church""","""Summer Street"""
"""7197-7195""","""7197-ee4b9f28""","""7195-565504a6""","""Parnell Library""","""Ayr Street"""
"""7213-7211""","""7213-c1b68c83""","""7211-824d717b""","""Ponsonby Road/Western Park""","""Ponsonby Road/Karangahape Road"""
"""8003-8001""","""8003-2ea7e096""","""8001-8ded30b5""","""Johnstone Street""","""Coyle Park"""
"""7028-1018""","""7028-f204fff8""","""1018-1ed97058""","""Fort Street""","""Woolworths Quay Street"""
"""8122-8124""","""8122-89235838""","""8124-c7d84961""","""Western Springs Park & MOTAT""","""Auckland Zoo"""
"""8064-8094""","""8064-d65bcd7e""","""8094-885d31e4""","""The Drive""","""Stop D Manukau Rd/Green Ln W"""
"""7830-7832""","""7830-8ced295b""","""7832-6a3a644b""","""Emerson Street""","""St Heliers Bay Road/West Tamak…"
"""7432-7434""","""7432-878c3d93""","""7434-ff8c4c9d""","""Blackett Crescent""","""Purewa Cemetery"""


## Table: `trips`

Scheduled trips, each linked to a route, service, and shape.


In [5]:
trips = query("SELECT * FROM trips")
print(trips.shape)
trips.head(10)

(6017, 6)


trip_id,route_id,service_id,direction_id,shape_id,created_at
str,str,str,i64,str,"datetime[μs, UTC]"
"""1278-02202-40200-2-de3d9f9f""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-25800-2-2b79cc90""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1095-19507-27300-2-958021f1""","""195-203""","""Weekday-2""",0,"""1095-19507-53d6d0f4""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-58200-2-b5f99905""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-47400-2-b5f99905""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1095-19507-49500-2-60d341d8""","""195-203""","""Weekday-2""",0,"""1095-19507-53d6d0f4""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-78000-2-323c18d3""","""22R-202""","""Sunday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-57300-2-d1c8c8b4""","""22R-202""","""Weekday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1095-19507-80100-2-99742a35""","""195-203""","""Sunday-2""",0,"""1095-19507-53d6d0f4""",2026-05-20 11:25:34.013059 UTC


## Table: `trip_segments`

Many-to-many mapping of trips to their constituent segments.


In [6]:
trip_segments = query("SELECT * FROM trip_segments")
print(trip_segments.shape)
trip_segments.head(10)

(57747, 2)


trip_id,segment_id
str,str
"""1285-07501-27000-2-5ed80f39""","""7445-7443"""
"""1285-07501-30600-2-5e056a4a""","""7425-7423"""
"""1285-07501-34200-2-f9f3ee0f""","""7429-7427"""
"""1285-07501-35100-2-5aa7f0c7""","""7439-7437"""
"""1285-07501-38700-2-10eb76c5""","""7443-7441"""
"""1285-07501-38700-2-5aa7f0c7""","""7437-7435"""
"""1285-07501-38700-2-5aa7f0c7""","""7423-7421"""
"""1285-07501-41400-2-8070c695""","""7445-7443"""
"""1285-07501-42300-2-5aa7f0c7""","""7437-7435"""


## Table: `vehicle_locations`

Real-time vehicle position feed — lat/lon, speed, delay, occupancy, etc.


In [7]:
vehicle_locations = query("""
    SELECT id, trip_id, vehicle_id, label, route_id, direction_id,
           latitude, longitude, bearing, speed, delay,
           occupancy_status, stop_id, stop_sequence,
           to_timestamp(timestamp) AS observed_at,
           start_time, created_at
    FROM vehicle_locations
    WHERE route_id LIKE '%65%'
    ORDER BY created_at DESC
""")
print(vehicle_locations.shape)
vehicle_locations.head(10)

(80247, 17)


id,trip_id,vehicle_id,label,route_id,direction_id,latitude,longitude,bearing,speed,delay,occupancy_status,stop_id,stop_sequence,observed_at,start_time,created_at
i64,str,str,str,str,i64,f64,f64,f64,f64,i64,i64,str,i64,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]"
15831,"""1076-06502-75600-2-2ec5b985""","""15831""","""NB5831""","""65-202""",1,-36.885595,174.740361,283.0,42.0,146,0,"""8049-df650ebd""",34,2026-05-30 09:29:39 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:30:00.884512 UTC
15777,"""1076-06506-76500-2-8a3901e0""","""15777""","""NB5777""","""65-202""",1,-36.889334,174.79651,238.0,32.0,-70,0,"""1718-b22b8c44""",23,2026-05-30 09:29:42 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:30:00.884512 UTC
15813,"""1076-06505-75600-2-0bf1c2ce""","""15813""","""NB5813""","""65-202""",0,-36.879508,174.818122,74.0,42.0,-10,1,"""7424-2c776042""",34,2026-05-30 09:29:40 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:30:00.884512 UTC
15938,"""1076-06501-76500-2-6fdfa012""","""15938""","""NB5938""","""65-202""",0,-36.887893,174.751644,106.0,43.0,46,1,"""8054-eef7810e""",17,2026-05-30 09:29:41 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:30:00.884512 UTC
15831,"""1076-06502-75600-2-2ec5b985""","""15831""","""NB5831""","""65-202""",1,-36.887203,174.747704,286.0,0.0,202,0,"""8053-24968585""",32,2026-05-30 09:28:14 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:29:00.897196 UTC
15777,"""1076-06506-76500-2-8a3901e0""","""15777""","""NB5777""","""65-202""",1,-36.888519,174.8003,239.0,21.0,-52,0,"""1718-b22b8c44""",23,2026-05-30 09:28:42 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:29:00.897196 UTC
15938,"""1076-06501-76500-2-6fdfa012""","""15938""","""NB5938""","""65-202""",0,-36.886966,174.747286,104.0,0.0,19,0,"""8050-fc9d077e""",15,2026-05-30 09:28:26 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:29:00.897196 UTC
15813,"""1076-06505-75600-2-0bf1c2ce""","""15813""","""NB5813""","""65-202""",0,-36.880532,174.811569,54.0,0.0,10,1,"""7422-beddf3d1""",33,2026-05-30 09:28:39 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:29:00.897196 UTC
15752,"""1076-06506-74700-2-8a3901e0""","""15752""","""NB5752""","""65-202""",1,-36.852745,174.704151,1.0,0.0,25,0,"""8001-8ded30b5""",54,2026-05-30 09:26:55 UTC,2026-05-30 08:45:00 UTC,2026-05-30 09:28:00.911065 UTC


In [8]:
# Summary stats for numeric columns
vehicle_locations.select(["speed", "delay", "bearing", "occupancy_status"]).describe()

statistic,speed,delay,bearing,occupancy_status
str,f64,f64,f64,f64
"""count""",80247.0,80247.0,80247.0,80247.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",20.40085,77.175122,187.441823,0.651052
"""std""",18.080996,303.118685,99.962022,0.590941
"""min""",0.0,-3555.0,0.0,0.0
"""25%""",0.0,-75.0,96.0,0.0
"""50%""",21.0,74.0,197.0,1.0
"""75%""",37.0,230.0,279.0,1.0
"""max""",94.0,2534.0,360.0,5.0


To do a network analysis, I need to assign vehicle locations to segments. I think the "stop id" is the last stop the bus was at, so assigning segments should be simple, but I need to validate this.


In [9]:
import folium
import webbrowser
import pathlib


m = folium.Map(location=[-36.848, 174.763], zoom_start=12)  # Auckland CBD

for row in vehicle_locations.head(1000).iter_rows(named=True):
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color="#e74c3c",
        fill=True,
        fill_opacity=0.8,
        tooltip=f"{row['stop_id']} | route {row['route_id']} | {row['speed']:.1f} m/s | delay {row['delay']}s",
    ).add_to(m)

m.save("map.html")
# webbrowser.open(pathlib.Path("map.html").resolve().as_uri())


Confirmed by noting busses on the way back to depot all have stop id = to the last stop on the trip. Also other spot checks

### What this means

This means that we can assign segments to vehicle locations based on their stop id, and their position in the trip


In [ ]:
trips

trip_id,route_id,service_id,direction_id,shape_id,created_at
str,str,str,i64,str,"datetime[μs, UTC]"
"""1278-02202-40200-2-de3d9f9f""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-25800-2-2b79cc90""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1095-19507-27300-2-958021f1""","""195-203""","""Weekday-2""",0,"""1095-19507-53d6d0f4""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-58200-2-b5f99905""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
"""1278-02202-47400-2-b5f99905""","""22R-202""","""Saturday-2""",1,"""1278-02202-cf7ff6d3""",2026-05-20 11:25:34.013059 UTC
…,…,…,…,…,…
"""1076-06502-40500-2-7e67457f""","""65-202""","""Weekday-1""",1,"""1076-06502-6181a08c""",2026-05-25 15:00:17.919246 UTC
"""1255-10501-40500-2-b20d629a""","""105-202""","""Sunday-1""",0,"""1255-10501-983a6559""",2026-05-25 15:00:17.919246 UTC
"""1285-07501-36000-2-d9bd5f01""","""75-202""","""Saturday-1""",0,"""1285-07501-c8719319""",2026-05-25 15:00:17.919246 UTC


In [11]:
stops

stop_id,location_type,stop_code,stop_lat,stop_lon,stop_name
str,i64,str,f64,f64,str
"""7890-7ac9c7e4""",0,"""7890""",-36.8567,174.81022,"""Paritai Drive"""
"""7309-2c09a359""",0,"""7309""",-36.84897,174.79291,"""Parnell Baths"""
"""8074-957bc7e4""",0,"""8074""",-36.88334,174.80648,"""Grand View Road"""
"""7108-87efa451""",0,"""7108""",-36.84815,174.7524,"""Franklin Road"""
"""8017-1a66e258""",0,"""8017""",-36.86701,174.70765,"""Alberta Street"""
…,…,…,…,…,…
"""7179-9b4fd003""",0,"""7179""",-36.84697,174.77318,"""Spark Arena"""
"""8099-114f1a71""",0,"""8099""",-36.86501,174.70549,"""Walker Park"""
"""7221-b28d532b""",0,"""7221""",-36.85039,174.74452,"""Ponsonby Methodist Church"""


In [12]:
segments

segment_id,start_stop_id,end_stop_id,start_stop_name,end_stop_name
str,str,str,str,str
"""7221-7219""","""7221-b28d532b""","""7219-06392a6a""","""Ponsonby Methodist Church""","""Summer Street"""
"""7197-7195""","""7197-ee4b9f28""","""7195-565504a6""","""Parnell Library""","""Ayr Street"""
"""7213-7211""","""7213-c1b68c83""","""7211-824d717b""","""Ponsonby Road/Western Park""","""Ponsonby Road/Karangahape Road"""
"""8003-8001""","""8003-2ea7e096""","""8001-8ded30b5""","""Johnstone Street""","""Coyle Park"""
"""7028-1018""","""7028-f204fff8""","""1018-1ed97058""","""Fort Street""","""Woolworths Quay Street"""
…,…,…,…,…
"""7223-7221""","""7223-bdcf6cc4""","""7221-b28d532b""","""Pompallier Terrace""","""Ponsonby Methodist Church"""
"""8074-7422""","""8074-957bc7e4""","""7422-beddf3d1""","""Grand View Road""","""Remuera Road/Remuera Village S…"
"""7888-7890""","""7888-6ecf54d7""","""7890-7ac9c7e4""","""Awarua Crescent""","""Paritai Drive"""


In [13]:
segments_wide = trip_segments.join(segments, on="segment_id")


In [14]:
vehicle_locations


id,trip_id,vehicle_id,label,route_id,direction_id,latitude,longitude,bearing,speed,delay,occupancy_status,stop_id,stop_sequence,observed_at,start_time,created_at
i64,str,str,str,str,i64,f64,f64,f64,f64,i64,i64,str,i64,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]"
15831,"""1076-06502-75600-2-2ec5b985""","""15831""","""NB5831""","""65-202""",1,-36.885595,174.740361,283.0,42.0,146,0,"""8049-df650ebd""",34,2026-05-30 09:29:39 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:30:00.884512 UTC
15777,"""1076-06506-76500-2-8a3901e0""","""15777""","""NB5777""","""65-202""",1,-36.889334,174.79651,238.0,32.0,-70,0,"""1718-b22b8c44""",23,2026-05-30 09:29:42 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:30:00.884512 UTC
15813,"""1076-06505-75600-2-0bf1c2ce""","""15813""","""NB5813""","""65-202""",0,-36.879508,174.818122,74.0,42.0,-10,1,"""7424-2c776042""",34,2026-05-30 09:29:40 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:30:00.884512 UTC
15938,"""1076-06501-76500-2-6fdfa012""","""15938""","""NB5938""","""65-202""",0,-36.887893,174.751644,106.0,43.0,46,1,"""8054-eef7810e""",17,2026-05-30 09:29:41 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:30:00.884512 UTC
15831,"""1076-06502-75600-2-2ec5b985""","""15831""","""NB5831""","""65-202""",1,-36.887203,174.747704,286.0,0.0,202,0,"""8053-24968585""",32,2026-05-30 09:28:14 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:29:00.897196 UTC
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14078,"""1076-06501-21000-2-75434e6b""","""14078""","""NB4078""","""65-202""",0,-36.883656,174.732771,124.0,32.0,214,1,"""8863-401d9e85""",11,2026-05-20 18:01:24 UTC,2026-05-20 17:50:00 UTC,2026-05-20 18:02:00.926742 UTC
14274,"""1076-06506-20700-2-b3c0f75a""","""14274""","""NB4274""","""65-202""",1,-36.88801,174.798789,231.0,47.0,152,0,"""1718-b22b8c44""",23,2026-05-20 18:00:22 UTC,2026-05-20 17:45:00 UTC,2026-05-20 18:01:00.949623 UTC
14078,"""1076-06501-21000-2-75434e6b""","""14078""","""NB4078""","""65-202""",0,-36.878838,174.72855,138.0,0.0,212,1,"""8863-401d9e85""",11,2026-05-20 18:00:24 UTC,2026-05-20 17:50:00 UTC,2026-05-20 18:01:00.949623 UTC


In [15]:
matched_locations = vehicle_locations.join(
    segments_wide,
    left_on=["trip_id", "stop_id"],
    right_on=["trip_id", "start_stop_id"],
    how="inner",
)
no_match = vehicle_locations.join(
    segments_wide,
    left_on=["trip_id", "stop_id"],
    right_on=["trip_id", "start_stop_id"],
    how="anti",
)

In [16]:
no_match["stop_id"].value_counts()

stop_id,count
str,u64
"""8098-a843ca65""",5109
"""8799-1d827315""",8227
"""8766-a2d9be31""",1505
"""8001-8ded30b5""",339


All instances of no match, were where the bus had finished its trip (either ends at the train station, or at one of three end stops in pt chev)


In [18]:
matched_locations

id,trip_id,vehicle_id,label,route_id,direction_id,latitude,longitude,bearing,speed,delay,occupancy_status,stop_id,stop_sequence,observed_at,start_time,created_at,segment_id,end_stop_id,start_stop_name,end_stop_name
i64,str,str,str,str,i64,f64,f64,f64,f64,i64,i64,str,i64,"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",str,str,str,str
15831,"""1076-06502-75600-2-2ec5b985""","""15831""","""NB5831""","""65-202""",1,-36.885595,174.740361,283.0,42.0,146,0,"""8049-df650ebd""",34,2026-05-30 09:29:39 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:30:00.884512 UTC,"""8049-8047""","""8047-fa0793b4""","""Bharatiya Mandir Temple""","""Balmoral Road/Sandringham Road"""
15777,"""1076-06506-76500-2-8a3901e0""","""15777""","""NB5777""","""65-202""",1,-36.889334,174.79651,238.0,32.0,-70,0,"""1718-b22b8c44""",23,2026-05-30 09:29:42 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:30:00.884512 UTC,"""1718-8069""","""8069-bd0dc0bc""","""Allevia Hospital Ascot""","""Green Lane West/Great South Ro…"
15813,"""1076-06505-75600-2-0bf1c2ce""","""15813""","""NB5813""","""65-202""",0,-36.879508,174.818122,74.0,42.0,-10,1,"""7424-2c776042""",34,2026-05-30 09:29:40 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:30:00.884512 UTC,"""7424-7426""","""7426-6b1b420c""","""Haast Street""","""Koraha Street"""
15938,"""1076-06501-76500-2-6fdfa012""","""15938""","""NB5938""","""65-202""",0,-36.887893,174.751644,106.0,43.0,46,1,"""8054-eef7810e""",17,2026-05-30 09:29:41 UTC,2026-05-30 09:15:00 UTC,2026-05-30 09:30:00.884512 UTC,"""8054-8056""","""8056-60e668ba""","""Matipo Street""","""Henley Road"""
15831,"""1076-06502-75600-2-2ec5b985""","""15831""","""NB5831""","""65-202""",1,-36.887203,174.747704,286.0,0.0,202,0,"""8053-24968585""",32,2026-05-30 09:28:14 UTC,2026-05-30 09:00:00 UTC,2026-05-30 09:29:00.897196 UTC,"""8053-8051""","""8051-0a88b3c6""","""Matipo Street""","""Balmoral Road/Dominion Road"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
14078,"""1076-06501-21000-2-75434e6b""","""14078""","""NB4078""","""65-202""",0,-36.883656,174.732771,124.0,32.0,214,1,"""8863-401d9e85""",11,2026-05-20 18:01:24 UTC,2026-05-20 17:50:00 UTC,2026-05-20 18:02:00.926742 UTC,"""8863-8044""","""8044-8c39a247""","""New North Road""","""Stop D St Lukes"""
14274,"""1076-06506-20700-2-b3c0f75a""","""14274""","""NB4274""","""65-202""",1,-36.88801,174.798789,231.0,47.0,152,0,"""1718-b22b8c44""",23,2026-05-20 18:00:22 UTC,2026-05-20 17:45:00 UTC,2026-05-20 18:01:00.949623 UTC,"""1718-8069""","""8069-bd0dc0bc""","""Allevia Hospital Ascot""","""Green Lane West/Great South Ro…"
14078,"""1076-06501-21000-2-75434e6b""","""14078""","""NB4078""","""65-202""",0,-36.878838,174.72855,138.0,0.0,212,1,"""8863-401d9e85""",11,2026-05-20 18:00:24 UTC,2026-05-20 17:50:00 UTC,2026-05-20 18:01:00.949623 UTC,"""8863-8044""","""8044-8c39a247""","""New North Road""","""Stop D St Lukes"""
